# 🏆 Comprehensive 8-Model Long-Text Generation Benchmark
### (`AGENTS' ROOM` • `STORYWRITER` • `Self-Lengthen` • `LongWriter-Llama3.1-8B` • `LongWriter-GLM4-9B` • `LongWriter-Zero-32B` • `MA-GRPO-50` • `MA-GRPO-100`)

Notebook này thiết lập **đầy đủ kiến trúc chuẩn, trọng số chính thức và định dạng Prompt gốc** của cả **8 mô hình**, chạy tuần tự trên Google Colab (hoặc VS Code / Cursor kết nối Colab Runtime / Local GPU) và xuất kết quả:

1. **8 File CSV chi tiết cho từng Model** (khớp **100% cấu trúc 12 cột** của `result_longwriter_llama.csv`):
   - `Model,Prompt_ID,Target_Len,Actual_Len,Length_Score_Sl (%),Distinct-3 (%),Distinct-4 (%),Latency (s),Peak_VRAM (GB),Speed (words/s),Prompt_Text,Raw_Output`
   - Danh sách file xuất ra:
     1. `result_agents_room.csv`
     2. `result_storywriter.csv`
     3. `result_self_lengthen.csv`
     4. `result_longwriter_llama.csv`
     5. `result_longwriter_glm.csv`
     6. `result_longwriter_zero.csv`
     7. `result_ma_grpo_50.csv`
     8. `result_ma_grpo_100.csv`

2. **Bảng Tổng Hợp So Sánh Benchmark Cuối Cùng (`benchmark_comparison_8models.csv`)**:
   - Tự động chấm **Perplexity (`PPL ↓`)** bằng `Qwen/Qwen2.5-0.5B` và **Overlap (`BLEU-4`, `ROUGE-L`)** so với truyện mẫu con người trong `tell-me-a-story-test.jsonl`.

---
### Điểm nâng cấp quan trọng so với bản chạy thử trước:
- **AGENTS' ROOM**: Khôi phục đầy đủ **9 Agent Roles** (4 Planning Sub-Agents: `[CONFLICT]`, `[CHARACTER]`, `[SETTING]`, `[PLOT]` ghi vào **Shared Scratchpad** + 5 Writing Sub-Agents theo **Freytag's Pyramid** có cân chỉnh ngân sách từ).
- **STORYWRITER**: Khôi phục đúng mã nguồn `StoryWriter/agent_try.py` gồm **Event Graph + Critic Rerank** $\to$ **Sub-Event Divider (3 sub-events/event)** $\to$ **Chapter Disperser** $\to$ **Chapter Writer kèm nén ngữ cảnh ReIO 10% (`MessageRedact`)**.
- **Self-Lengthen**: Khôi phục đúng thuật toán `step1` + `step2_ext_responses.py` với template `extend_response.md`, cắt $2/3$ đoạn đầu và nối tiếp bằng **Raw Prefix Completion (`get_completion`)** kèm bộ lọc lặp 10-gram `check_response`.
- **LongWriter-Zero-32B (`THU-KEG/LongWriter-Zero-32B`)**: Chạy **thật 100%** trọng số 32B bằng **4-bit NF4 Double Quantization (`~18.2 GB VRAM`)** + `low_cpu_mem_usage=True` (tự động offload nếu GPU < 21GB để không bao giờ bị OOM) và cấp đủ `max_new_tokens >= 3600` cho cả chuỗi suy nghĩ `<think>...</think>` lẫn phần truyện `<answer>...</answer>`.

## Bước 1: Kiểm tra phần cứng GPU & Kết nối Google Drive

In [ ]:
!nvidia-smi

import os
import torch

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f" -> GPU {i}: {name} ({vram:.2f} GB VRAM)")

# Tự động kết nối Google Drive nếu chạy trên Colab
IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    OUTPUT_DIR = "/content/drive/MyDrive/MA-GRPO-Checkpoints/benchmark_8models_output"
    CKPT50_PATH = "/content/drive/MyDrive/MA-GRPO-Checkpoints/ma-grpo-final/checkpoint-50"
    CKPT100_PATH = "/content/drive/MyDrive/MA-GRPO-Checkpoints/ma-grpo-final/checkpoint-100"
except ImportError:
    OUTPUT_DIR = os.path.abspath("./benchmark_8models_output")
    CKPT50_PATH = "C:/Users/ASUS/AI_LLMs/Long_text/ma-grpo-final-20261029T095313Z-1-001/ma-grpo-final/checkpoint-50"
    CKPT100_PATH = "C:/Users/ASUS/AI_LLMs/Long_text/ma-grpo-final-20261029T095313Z-1-001/ma-grpo-final/checkpoint-100"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Thư mục lưu kết quả CSV: {OUTPUT_DIR}")

## Bước 2: Clone / Đồng bộ mã nguồn `MA-GRPO` (kèm thư mục `Comparison/`) & Cài đặt thư viện

In [ ]:
import os
import sys
import getpass

if IS_COLAB:
    REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
    if not os.path.exists("/content/MA-GRPO"):
        try:
            from google.colab import userdata
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("Nhập GitHub Personal Access Token (nếu repo private, hoặc Enter nếu public): ")
        if token:
            !git clone https://{token}@{REPO_URL} /content/MA-GRPO
        else:
            !git clone https://{REPO_URL} /content/MA-GRPO
    else:
        %cd /content/MA-GRPO
        !git pull
    %cd /content/MA-GRPO
    if "/content/MA-GRPO" not in sys.path:
        sys.path.insert(0, "/content/MA-GRPO")
else:
    # Nếu chạy ở Local VS Code / Cursor
    local_repo = os.path.abspath("C:/Users/ASUS/AI_LLMs/Long_text/MA-GRPO")
    if os.path.exists(local_repo):
        os.chdir(local_repo)
        if local_repo not in sys.path:
            sys.path.insert(0, local_repo)
    print(f"Working directory: {os.getcwd()}")

!pip install -q --upgrade transformers>=4.45.0 accelerate>=0.33.0 peft>=0.12.0 bitsandbytes>=0.43.0 hf-transfer nltk rouge-score pandas
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("✅ Đã cài đặt xong các thư viện và kích hoạt hf-transfer!")

## Bước 3: Nạp tập dữ liệu Benchmark (`tell-me-a-story-test.jsonl`)
- Mặc định `NUM_SAMPLES = 3` (`example_000`: 1350 từ, `example_001`: 1575 từ, `example_002`: 1900 từ).
- Nếu muốn chạy đủ 5 mẫu như `result_longwriter_llama.csv`, đổi `NUM_SAMPLES = 5`.

In [ ]:
from pathlib import Path
from Comparison.metrics import load_benchmark_prompts
from Comparison.qwen_backend import QwenAgentBackend

NUM_SAMPLES = 3  # Đổi thành 5 nếu muốn chạy cả 5 prompts (example_000 -> example_004)
CLEAR_HF_CACHE_BETWEEN_HEAVY_MODELS = False  # Đặt True nếu ổ đĩa Colab còn ít dung lượng trống (<35GB)

prompts = load_benchmark_prompts(num_samples=NUM_SAMPLES)
print(f"✅ Đã nạp {len(prompts)} prompts chuẩn:")
for p in prompts:
    print(f"  - {p['id']}: Target = {p['target_length']} words | Prompt: {p['prompt'][:90]}...")

---
## NHÓM 1: MULTI-AGENT & ITERATIVE SELF-ALIGNMENT BASELINES
*(Dùng chung bộ não `Qwen/Qwen2.5-7B-Instruct` nạp 1 lần vào VRAM cho cả 3 phương pháp `AGENTS' ROOM`, `STORYWRITER`, `Self-Lengthen`)*

### Model 1/8: `AGENTS' ROOM` (Google DeepMind, 2024)
- **Kiến trúc**: 4 Planning Sub-Agents (`[CONFLICT]`, `[CHARACTER]`, `[SETTING]`, `[PLOT]`) ghi vào **Shared Scratchpad** + 5 Writing Sub-Agents nối tiếp theo **Freytag's Pyramid** (`Exposition` $\to$ `Rising Action` $\to$ `Climax` $\to$ `Falling Action` $\to$ `Resolution`).
- **File xuất ra**: `result_agents_room.csv`

In [ ]:
from Comparison.agents_room import run_agents_room_benchmark

shared_qwen_backend = QwenAgentBackend("Qwen/Qwen2.5-7B-Instruct", load_in_4bit=True)
shared_qwen_backend.load()

df_agents_room = run_agents_room_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_agents_room.csv",
    backend=shared_qwen_backend,
)

### Model 2/8: `STORYWRITER` (THU-KEG, 2025)
- **Kiến trúc**: Khôi phục đúng mã nguồn `StoryWriter/agent_try.py`:
  1. `event_model` + `event_rerank_model` (Sinh đồ thị sự kiện 6 trường & Critic kiểm duyệt mạch truyện).
  2. `sub_events` (Phân rã mỗi Event thành 3 Sub-events `Event 1.1, 1.2, 1.3`).
  3. `disperse_events` (Phân bổ Sub-events vào các chương).
  4. `story_writer` + `MessageRedact` **ReIO 10% Condensation** (Nén các chương trước xuống 10% dung lượng giữ lại cốt truyện & xung đột chính).
- **File xuất ra**: `result_storywriter.csv`

In [ ]:
from Comparison.storywriter import run_storywriter_benchmark

df_storywriter = run_storywriter_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_storywriter.csv",
    backend=shared_qwen_backend,
)

### Model 3/8: `Self-Lengthen` (Alibaba / Qwen Team, 2024)
- **Kiến trúc**: Khôi phục đúng `step1_gen_initial_responses.py` & `step2_ext_responses.py`:
  1. **Generator**: Sinh bản thảo khởi tạo ban đầu.
  2. **Extender (Iterative Split-and-Extend)**: Cắt nửa đầu (`former`), mở rộng bằng template `extend_response.md`, giữ lại $2/3$ số đoạn của `former_output`, và dùng **Raw Prefix Completion (`get_completion`)** để viết nối mượt mà phần `latter_output`, kiểm duyệt bằng hàm `check_response` (lọc lặp 10-gram & dấu câu).
- **File xuất ra**: `result_self_lengthen.csv`

In [ ]:
from Comparison.self_lengthen import run_self_lengthen_benchmark

df_self_lengthen = run_self_lengthen_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_self_lengthen.csv",
    backend=shared_qwen_backend,
)

# Giải phóng VRAM của shared_qwen_backend trước khi nạp dòng LongWriter
shared_qwen_backend.unload()
del shared_qwen_backend

---
## NHÓM 2: HỌ MÔ HÌNH LONGWRITER (SFT / DPO / PURE RL GRPO 32B)

### Model 4/8: `LongWriter-llama3.1-8b` (`THUDM/LongWriter-llama3.1-8b`)
- **Trọng số chính thức**: `THUDM/LongWriter-llama3.1-8b`.
- **Định dạng chuẩn tác giả (`LongWriter/evaluation/pred.py`)**: `[INST]{prompt}[/INST]`, `temperature=0.5`, `top_p=0.9`, `repetition_penalty=1.05`.
- **File xuất ra**: `result_longwriter_llama.csv`

In [ ]:
from Comparison.longwriter_models import run_longwriter_llama_benchmark

df_lw_llama = run_longwriter_llama_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_longwriter_llama.csv",
    clear_hf_cache_after=CLEAR_HF_CACHE_BETWEEN_HEAVY_MODELS,
)

### Model 5/8: `LongWriter-glm4-9b` (`THUDM/LongWriter-glm4-9b`)
- **Trọng số chính thức**: `THUDM/LongWriter-glm4-9b` (tự động fallback `zai-org/LongWriter-glm4-9b`).
- **Bản vá tương thích**: Tích hợp đầy đủ bản vá `ChatGLMConfig.max_length`, `DynamicCache.__getitem__` và `all_tied_weights_keys` để chạy với `use_cache=True` tốc độ cao.
- **File xuất ra**: `result_longwriter_glm.csv`

In [ ]:
import importlib
from transformers.configuration_utils import PretrainedConfig
PretrainedConfig._get_generation_parameters = lambda self, *args, **kwargs: {}

import Comparison.longwriter_models
importlib.reload(Comparison.longwriter_models)
from Comparison.longwriter_models import run_longwriter_glm_benchmark

df_lw_glm = run_longwriter_glm_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_longwriter_glm.csv",
    clear_hf_cache_after=CLEAR_HF_CACHE_BETWEEN_HEAVY_MODELS,
)

### Model 6/8: `LongWriter-Zero-32B` (`THU-KEG/LongWriter-Zero-32B`)
- **Trọng số chính thức**: `THU-KEG/LongWriter-Zero-32B` (32.8B tham số, huấn luyện Pure RL GRPO trên nền Qwen2.5-32B).
- **Thiết lập chuẩn tránh OOM & Cắt cụt**:
  1. Lượng tử hóa **4-bit NF4 Double Quantization** + `low_cpu_mem_usage=True` giúp giảm dung lượng VRAM xuống **~18.2 GB** (chạy 100% trên GPU A100 40GB / L4 24GB / Dual T4 30GB; tự động bật GPU+CPU offload nếu chạy trên Single T4 15GB).
  2. Cấp `max_new_tokens = max(3600, int(target_len * 2.25))` để mô hình sinh trọn vẹn chuỗi suy nghĩ `<think>...</think>` (~800–1200 tokens) VÀ toàn bộ câu chuyện trong `<answer>...</answer>`.
  3. Tự động bóc tách phần truyện sạch bên ngoài `<think>` để chấm điểm công bằng 100%.
- **File xuất ra**: `result_longwriter_zero.csv`

In [ ]:
import transformers.modeling_utils

def _get_safe_tied(self):
    val = self.__dict__.get("_custom_all_tied_weights_keys", None)
    if val is not None and hasattr(val, "keys"):
        return val
    val2 = getattr(self, "_tied_weights_keys", None)
    return val2 if (val2 is not None and hasattr(val2, "keys")) else {}

def _set_safe_tied(self, value):
    self.__dict__["_custom_all_tied_weights_keys"] = value if (value is not None and hasattr(value, "keys")) else {}

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(_get_safe_tied, _set_safe_tied)

from Comparison.longwriter_models import run_longwriter_zero_benchmark

df_lw_zero = run_longwriter_zero_benchmark(
    prompts=prompts,
    output_csv=Path(OUTPUT_DIR) / "result_longwriter_zero.csv",
    clear_hf_cache_after=CLEAR_HF_CACHE_BETWEEN_HEAVY_MODELS,
)

---
## NHÓM 3: MÔ HÌNH ĐỀ XUẤT `MA-GRPO` (`checkpoint-50` & `checkpoint-100`)

### Model 7/8: `MA-GRPO-50` (Checkpoint 50 Steps)
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4) + LoRA Adapter `checkpoint-50`.
- **Cấu trúc Prompt**: Phân bổ ngân sách 3 cảnh (`Scene 1: 25%`, `Scene 2: 45%`, `Scene 3: 30%`) + lập kế hoạch Test-Time Reasoning `<think>...</think>` trước khi viết truyện.
- **File xuất ra**: `result_ma_grpo_50.csv`

In [ ]:
from Comparison.ma_grpo_runner import run_ma_grpo_checkpoint_benchmark

df_magrpo_50 = run_ma_grpo_checkpoint_benchmark(
    prompts=prompts,
    step=50,
    output_csv=Path(OUTPUT_DIR) / "result_ma_grpo_50.csv",
    checkpoint_dir=CKPT50_PATH,
)

### Model 8/8: `MA-GRPO-100` (Checkpoint 100 Steps)
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4) + LoRA Adapter `checkpoint-100`.
- **File xuất ra**: `result_ma_grpo_100.csv`

In [ ]:
df_magrpo_100 = run_ma_grpo_checkpoint_benchmark(
    prompts=prompts,
    step=100,
    output_csv=Path(OUTPUT_DIR) / "result_ma_grpo_100.csv",
    checkpoint_dir=CKPT100_PATH,
)

---
## 📊 Bước Cuối: Tổng hợp 8 File CSV & Xuất Bảng Đối Sánh Benchmark Toàn Diện
- Đọc toàn bộ 8 file `result_*.csv` vừa sinh.
- Đo **Perplexity (`PPL ↓`)** bằng `Qwen/Qwen2.5-0.5B` và tính **BLEU-4 / ROUGE-L** đối chiếu với truyện gốc trong `tell-me-a-story-test.jsonl`.
- Xuất bảng tổng hợp `benchmark_comparison_8models.csv` và `benchmark_results_all_raw.csv`.

In [ ]:
from IPython.display import display
from Comparison.run_all_benchmarks import build_final_comparison_table

df_final_summary = build_final_comparison_table(
    output_dir=Path(OUTPUT_DIR),
    prompts=prompts,
    summary_csv_name="benchmark_comparison_8models.csv",
)

display(df_final_summary)